### Get icd code YAML file

In [ ]:
#import libraries 

import pandas as pd
import openpyxl
import duckdb 
import pyarrow.parquet as pq

In [ ]:
# load and read icd codes datasets 

icd9_file = pd.read_excel('../data/input/medpar_outcomes_icd_codes/icd9_codes.xlsx')
icd10_file = pd.read_excel('../data/input/medpar_outcomes_icd_codes/icd10_codes.xlsx')

In [ ]:
# Function to match ICD codes

def match_codes(df, codes):
    pattern = f"^(?:{'|'.join(codes)})"
    matched = df[df['CODE'].astype(str).str.contains(pattern, regex=True)]
    return matched['CODE'].unique()

In [ ]:
conditions = {
    "Cardiovascular Diseases": {
        "icd9": [str(i) for i in range(390, 460)],
        "icd10": [f"I{i:02d}" for i in range(0, 100)]
    },
    "Stroke": {
        "icd9": [str(i) for i in range(431, 435)] + ["436"],
        "icd10": ["I60", "I61", "I63", "I64"]
    },
    "Ischemic Stroke": {
        "icd9": ["433", "434"],
        "icd10": ["I63"]
    },
    "Myocardial Infarction": {
        "icd9": ["410"],
        "icd10": ["I21"]
    },
    "Heart Failure": {
        "icd9": ["428"],
        "icd10": ["I50"]
    },
    "Respiratory Diseases": {
        "icd9": [str(i) for i in range(460, 520)],
        "icd10": [f"J{i:02d}" for i in range(0, 100)]
    },
    "Chronic Obstructive Pulmonary Disease": {
        "icd9": ["491", "492", "496"],
        "icd10": ["J41", "J42", "J43", "J44"]
    },
    "Pneumonia": {
        "icd9": [str(i) for i in range(480, 487)],
        "icd10": [f"J{i}" for i in range(12, 19)]
    },
    "Acute Bronchitis": {
        "icd9": ["4660"],
        "icd10": ["J20"]
    },
    "Acute Kidney Injury": {
        "icd9": ["584"],
        "icd10": ["N17"]
    },
    "Fluid and Electrolyte Disorders": {
        "icd9": ["276"],
        "icd10": ["E86", "E87"]
    },
    "Intestinal Infectious Diseases": {
        "icd9": [f"{i:03d}" for i in range(1, 10)],
        "icd10": [f"A{i:02d}" for i in range(0, 10)]
    }
}


In [ ]:
short_names = {
    "Cardiovascular Diseases": "cvd",
    "Stroke": "stroke",
    "Ischemic Stroke": "ischemic_stroke",
    "Myocardial Infarction": "myocardial_infarction",
    "Heart Failure": "heart_failure",
    "Respiratory Diseases": "respiratory_diseases",
    "Chronic Obstructive Pulmonary Disease": "copd",
    "Pneumonia": "pneumonia",
    "Acute Bronchitis": "acute_bronchitis",
    "Acute Kidney Injury": "acute_kidney_injury",
    "Fluid and Electrolyte Disorders": "fluid_electrolyte_disorders",
    "Intestinal Infectious Diseases": "intestinal_infectious_diseases",
}


In [ ]:
# write the outcomes names, icd codes 9 and 10 , outcome_criteria and readmission window days (if needed) to reflect in the yaml output of teh respective project 

yaml_output = " "

for condition, codes in conditions.items():

    identifier = short_names[condition]

    # Match ICD-9 and ICD-10 codes
    icd9_codes = match_codes(icd9_file, codes['icd9']) if codes['icd9'] else []
    icd10_codes = match_codes(icd10_file, codes['icd10']) if codes['icd10'] else []

    yaml_section = f"""
{identifier} :
    long_name : "{condition}"
    outcome_criteria: first_n
    first_n: 1
    icd9 : {list(icd9_codes)}
    icd9_count: {len(icd9_codes)}
    icd10 : {list(icd10_codes)}
    icd10_count: {len(icd10_codes)}
"""
    yaml_output += yaml_section

with open('../conf/icd_codes/michelle_yichen_00.yml', 'w') as file:
    file.write(yaml_output)

print(yaml_output)